In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

from movebench.study import Study
from movebench.ssf import grouped_nll, iSSFDataset, get_issf_channel_info, MOVEMENT_FEATURES

In [3]:
study = Study.load('tsavo_lions') # ('5a280544-06db-4c85-9713-842547972edd')
study.download_rasters(splits=["train", "test"],
                        skip_existing=False
                        )

print(study)

[download_study_rasters] Study: tsavo_lions
[download_study_rasters] Total points: 2154
[download_study_rasters] Splits: ['train', 'test'], Chip groups: ['era5', 'modis', 'high_res'], Tile layers: ['dem', 'dist_water']

[download_study_rasters] Downloading train chips (1330 points)...
[download_chips] Buffer: 19.939 km, split: train, groups: ['era5', 'modis', 'high_res']
[download_chips] Starting download of 1330 chips with 16 workers


[download_chips] Complete: 3990 downloaded, 0 existing, 0 failed

[download_study_rasters] Downloading test chips (824 points)...
[download_chips] Buffer: 19.939 km, split: test, groups: ['era5', 'modis', 'high_res']
[download_chips] Starting download of 824 chips with 16 workers


[download_chips] Complete: 2472 downloaded, 0 existing, 0 failed

[download_study_rasters] Downloading dem tiles...
[download_tile_layer:dem] Buffer: 19.939 km, scale: 398.8m
[download_tile_layer:dem] Creating union of 2154 regions...
[download_tile_layer:dem] Created 2 tiles
[download_tile_layer:dem] Downloading tile 1/2
[download_tile_layer:dem] Downloading tile 2/2
[download_tile_layer:dem] Complete: 2 downloaded, 0 existing, 0 failed

[download_study_rasters] Downloading dist_water tiles...
[download_tile_layer:dist_water] Buffer: 19.939 km, scale: 398.8m
[download_tile_layer:dist_water] Creating union of 2154 regions...
[download_tile_layer:dist_water] Created 2 tiles
[download_tile_layer:dist_water] Downloading tile 1/2
[download_tile_layer:dist_water] Downloading tile 2/2
[download_tile_layer:dist_water] Complete: 2 downloaded, 0 existing, 0 failed

[download_study_rasters] All downloads complete for tsavo_lions
Study('tsavo_lions')


In [ ]:
# Create iSSF dataset
issf_ds = study.issf_dataset(split='train',
                             include_time=True)
print(f"iSSF dataset: {len(issf_ds)} samples")
print(f"Implementation: {type(issf_ds).__module__}.{type(issf_ds).__name__}")

# Load one sample
step_pixels, valid_pixels, transform, crs = issf_ds[0]
print(f"Step pixels shape: {step_pixels.shape}")

# Show channel layout
info = get_issf_channel_info()
print(f"\nChannel layout:")
print(f"  Env features:   channels {info['env'][0]}-{info['env'][1]-1} ({info['n_env']} channels)")
print(f"  sl_:            channel {info['sl_'][0]}")
print(f"  log_sl_:        channel {info['log_sl_'][0]}")
print(f"  cos_ta_:        channel {info['cos_ta_'][0]}")
print(f"  Interactions:   channels {info['interactions'][0]}-{info['interactions'][1]-1} ({info['n_interactions']} channels)")
print(f"  Total:          {info['total']} channels")

[movebench] Computing normalization for 22 layers: ['modis_lst', 'modis_vi', 'modis_ndsi', 'modis_bai', 'era5_land', 'era5', 'era5_land_daily', 'era5_daily', 'dem', 'slope', 'aspect', 'rugosity', 'dist_water', 'orographic_uplift', 'thermal_uplift', 'modis_cvf_tree', 'modis_cvf_nontree', 'modis_cvf_nonveg', 'hii', 'bioclim', 'dynamic_world', 'landsat_ndwi']


Computing stats:   5%|▌         | 53/1000 [00:33<08:27,  1.86it/s]

In [ ]:
# Inspect movement features for the true endpoint (last column)
print("Movement features for true endpoint (index 100):")
print(f"  sl_     = {step_pixels[info['sl_'][0], -1].item():.2f} m")
print(f"  log_sl_ = {step_pixels[info['log_sl_'][0], -1].item():.4f}")
print(f"  cos_ta_ = {step_pixels[info['cos_ta_'][0], -1].item():.4f}")

print(f"\nMovement features for first pseudo-candidate (index 0):")
print(f"  sl_     = {step_pixels[info['sl_'][0], 0].item():.2f} m")
print(f"  log_sl_ = {step_pixels[info['log_sl_'][0], 0].item():.4f}")
print(f"  cos_ta_ = {step_pixels[info['cos_ta_'][0], 0].item():.4f}")

In [ ]:
# Check validity
move_start, move_end = info['movement']
print(f"Movement validity (should be all True): {valid_pixels[move_start:move_end].all().item()}")
print(f"Overall valid ratio: {valid_pixels.float().mean().item():.2%}")

---
## Train iSSF Model

Train a conditional logistic regression model using the iSSFDataset.
The model learns coefficients for environmental features, movement features,
and their interactions.

In [ ]:
# Extract features from iSSFDataset

def extract_ssf_features(dataset, num_workers=8):
    """Extract all features from an SSFDataset into tensors."""
    loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=num_workers)
    
    all_xs = []
    all_masks = []
    
    for step_pixels, valid_pixels, _, _ in tqdm(loader, desc="Extracting features"):
        all_xs.append(step_pixels)
        all_masks.append(valid_pixels)
    
    # Stack: (N, C, num_candidates+1)
    X = torch.cat(all_xs, dim=0)
    masks = torch.cat(all_masks, dim=0)
    
    # Reshape to (N * (num_candidates+1), C) for regression
    N, C, K = X.shape
    X = X.permute(0, 2, 1).reshape(-1, C)
    masks = masks.permute(0, 2, 1).reshape(-1, C)
    
    # Labels: last candidate (index K-1) is the true endpoint
    y = torch.zeros(N * K, dtype=torch.long)
    y[K-1::K] = 1  # Every K-th element starting from K-1
    
    # Group indices for conditional logistic regression
    starts = torch.arange(0, N * K, K)
    ends = torch.arange(K, N * K + 1, K)
    
    return X, masks, y, starts, ends

# Extract train features
train_issf = study.issf_dataset(split='train')

X_tr, masks_tr, y_tr, starts_tr, ends_tr = extract_ssf_features(train_issf)

print(f"Train features: {X_tr.shape}")
print(f"Train groups: {len(starts_tr)}")

In [ ]:
# Clean features: fill invalid values with 0 (approx mean after normalization)
def clean_features(X, masks):
    X_clean = torch.where(masks, X, torch.tensor(0.0))
    X_clean = torch.nan_to_num(X_clean, nan=0.0, posinf=0.0, neginf=0.0)
    return X_clean

X_tr_clean = clean_features(X_tr, masks_tr)

print(f"Features per sample: {X_tr_clean.shape[1]}")

In [ ]:
# Train conditional logistic regression
num_features = X_tr_clean.shape[1]
beta = torch.zeros(num_features, dtype=torch.float32, requires_grad=True)
opt = torch.optim.Adam([beta], 
                       lr=1e-2, 
                    #    weight_decay=1e-3
                       )

for step in range(5000):
    opt.zero_grad()
    loss = grouped_nll(beta, X_tr_clean, y_tr, starts_tr, ends_tr)
    loss.backward()
    opt.step()
    
    if step % 100 == 0:
        with torch.no_grad():
            nll_tr = grouped_nll(beta, X_tr_clean, y_tr, starts_tr, ends_tr).item()
            print(f"Step {step:4d} | Train NLL: {nll_tr:.4f}")

In [ ]:
# Inspect learned coefficients
from movebench.config import get_channel_indices, LAYERS

# Build channel names matching the iSSF layout
indices = get_channel_indices()
env_channel_names = []
for name, (start, end) in indices.items():
    layer = LAYERS[name]
    if layer.categorical:
        for i in range(layer.num_classes):
            env_channel_names.append(f"{name}_class{i}")
    else:
        for band in layer.bands:
            env_channel_names.append(f"{name}_{band}")

# Movement feature names
move_names = list(MOVEMENT_FEATURES)

# Interaction names
interact_names = []
for mf in MOVEMENT_FEATURES:
    for env_name in env_channel_names:
        interact_names.append(f"{mf}:{env_name}")

channel_names = env_channel_names + move_names + interact_names

beta_np = beta.detach().numpy()
sorted_idx = abs(beta_np).argsort()[::-1]

print("Top 15 coefficients by magnitude:\n")
for i in sorted_idx[:15]:
    print(f"{channel_names[i]:45s} {beta_np[i]:+.4f}")

print("\n--- Movement feature coefficients ---")
info = get_issf_channel_info()
move_start, move_end = info['movement']
for i in range(move_start, move_end):
    print(f"{channel_names[i]:45s} {beta_np[i]:+.4f}")

## Test Set Evaluation

In [ ]:
import time

# Time test set extraction
test_issf = study.issf_dataset(split='test',
                               include_time=True)
print(f"Test dataset: {len(test_issf)} samples")

start = time.time()
X_te, masks_te, y_te, starts_te, ends_te = extract_ssf_features(test_issf)
elapsed = time.time() - start

print(f"Test extraction time: {elapsed:.1f}s")
print(f"Test features: {X_te.shape}")

In [ ]:
# Evaluate on test set
X_te_clean = clean_features(X_te, masks_te)

with torch.no_grad():
    nll_te = grouped_nll(beta, X_te_clean, y_te, starts_te, ends_te).item()
    print(f"Test NLL: {nll_te:.4f}")